In [ ]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.pipeline import run_pipeline_all_years

In [ ]:
def build_Mcp_non_binarized_from_BACI(
    BACI_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Input:
    - BACI_df: dataframe BACI relativo a UN SOLO anno, con export/import per prodotto per ogni nazione 

    Output:
    - Matrice Mcp (country x product), valori reali
    
    """
    df = BACI_df.copy()

    country_tot = df.groupby('country')['v'].transform('sum')
    product_tot = df.groupby('k')['v'].transform('sum')
    total = df['v'].sum()

    df['Mcp'] = (df['v'] / country_tot) / (product_tot / total)
    

    matrix = df.pivot_table(
        index= 'country',
        columns= 'k',
        values='Mcp',
        fill_value=0
    ).astype(float)
    return matrix

In [ ]:
BACI_df = pd.read_parquet(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\BACI_export\BACI_export_HS02_Y2002.parquet')

In [ ]:
matrix = build_Mcp_non_binarized_from_BACI(BACI_df)     

In [ ]:
def Mcp_from_BACI_pipeline(
    BACI_folder: str,
    Mcp_folder: str,
) :
    """
    Converte tutti i file di una cartella in Mcp
    
    Input:
    - BACI_folder: cartella con i file BACI
    - Mcp_folder: cartella in cui salvare i file Mcp
    
    """
    os.makedirs(Mcp_folder, exist_ok=True)

    for filename in os.listdir(BACI_folder):
        if not filename.endswith(".parquet"):
            continue

        BACI_path = os.path.join(BACI_folder, filename)
        BACI_df = pd.read_parquet(BACI_path)

        Mcp = build_Mcp_non_binarized_from_BACI(BACI_df)

        Mcp_path = os.path.join(Mcp_folder, filename)
        Mcp.to_parquet(Mcp_path, index=False)

In [ ]:
BACI_folder = r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\BACI_export'
Mcp_folder = r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\Mcp_export_non_binarized'

In [ ]:
Mcp_from_BACI_pipeline(BACI_folder,Mcp_folder)

### ALLINEAMENTO

In [8]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import run_pipeline_all_years_completa

In [17]:
output_folder = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\alligned\binarized\Mcp_import') 
trade_type = "import"
Mcp_folder = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\non_alligned\binarized\Mcp_import')
BACI_folder = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\non_alligned\binarized\BACI_import')
C = pd.read_parquet(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\non_alligned\binarized\C\C_4_digits_HS02.parquet')

In [18]:
Mcp = run_pipeline_all_years_completa(Mcp_folder, BACI_folder, C, output_folder, trade_type, binarized = True)

  [Mcp  vs  C]
     Codici in Mcp: 1244  |  Codici in C: 1205
     In comune: 1205
     Solo in Mcp (mancanti in C): 39 -> ['0508', '9401', '9402', '9403', '9404', '9405', '9406', '9501', '9502', '9503', '9504', '9505', '9506', '9507', '9508', '9601', '9602', '9603', '9604', '9605', '9606', '9607', '9608', '9609', '9610', '9611', '9612', '9613', '9614', '9615', '9616', '9617', '9618', '9701', '9702', '9703', '9704', '9705', '9706']
     Solo in C (mancanti in Mcp): 0 -> []
Export codici selezionati: 170316963.963
Export totale: 6493062526.625
Quota: 0.026230605860425665
  [Mcp  vs  C]
     Codici in Mcp: 1244  |  Codici in C: 1205
     In comune: 1205
     Solo in Mcp (mancanti in C): 39 -> ['0508', '9401', '9402', '9403', '9404', '9405', '9406', '9501', '9502', '9503', '9504', '9505', '9506', '9507', '9508', '9601', '9602', '9603', '9604', '9605', '9606', '9607', '9608', '9609', '9610', '9611', '9612', '9613', '9614', '9615', '9616', '9617', '9618', '9701', '9702', '9703', '9704', '97

### CAMBIO NOME FILE

In [ ]:
import os

In [ ]:
def rinomina_file(cartella, vecchio_prefisso="BACI", nuovo_prefisso="Mcp"):
    for nome_file in os.listdir(cartella):
        percorso_vecchio = os.path.join(cartella, nome_file)
        
        if nome_file.startswith(vecchio_prefisso):
            nuovo_nome = nuovo_prefisso + nome_file[len(vecchio_prefisso):]
            percorso_nuovo = os.path.join(cartella, nuovo_nome)
            os.rename(percorso_vecchio, percorso_nuovo)
            print(f"Rinominato: {nome_file} -> {nuovo_nome}")



In [ ]:
cartella = r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\non_alligned\Mcp_import_non_binarized'

In [ ]:
vecchio_prefisso = "BACI"
nuovo_prefisso = "Mcp"

In [ ]:
os.listdir(cartella)

In [ ]:
for nome_file in os.listdir(cartella):
    # modifica nomi dei file singoli
    percorso_vecchio = os.path.join(cartella, nome_file)
    nuovo_nome = nuovo_prefisso + nome_file[len(vecchio_prefisso):]
    print (nuovo_nome)  
    # questa mi serve solo perchè os agisce sul path completo non sui singoli files
    percorso_nuovo = os.path.join(cartella, nuovo_nome)
    os.rename(percorso_vecchio, percorso_nuovo)

### Correzione errore codici HS6 import, non alligned, binarized

In [1]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import Mcp_from_BACI_pipeline

In [6]:
BACI_folder = r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\non_alligned\binarized\BACI_export'
Mcp_folder = r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\non_alligned\binarized\Mcp_export'

In [7]:
Mcp_from_BACI_pipeline(BACI_folder, Mcp_folder)